# 📊 Sales & Revenue Analysis Dashboard
**Author:** Aakriti Kumari | B.Tech CSE Student

**Objective:** Analyze sales performance, identify revenue trends, and generate key business insights using Python.

---

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('✅ Libraries loaded successfully!')

## Step 2: Load & Explore Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('../data/sales_data.csv')

print('Shape:', df.shape)
print('\nColumn Names:', df.columns.tolist())
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Basic info
print('Dataset Info:')
df.info()
print('\nNull Values:')
print(df.isnull().sum())
print('\nStatistical Summary:')
df.describe()

## Step 3: Data Cleaning & Feature Engineering

In [ ]:
# Convert Order_Date to datetime
df['Order_Date'] = pd.to_datetime(df['Order_Date'])

# Extract time-based features
df['Month'] = df['Order_Date'].dt.month
df['Month_Name'] = df['Order_Date'].dt.strftime('%b')
df['Quarter'] = df['Order_Date'].dt.quarter.map({1:'Q1', 2:'Q2', 3:'Q3', 4:'Q4'})
df['Year'] = df['Order_Date'].dt.year

# Calculate Profit Margin
df['Profit_Margin'] = (df['Profit'] / df['Sales_Amount'] * 100).round(2)

print('✅ Feature engineering done!')
df[['Order_Date','Month_Name','Quarter','Profit_Margin']].head()

## Step 4: KPI Calculations

In [ ]:
total_revenue    = df['Sales_Amount'].sum()
total_profit     = df['Profit'].sum()
total_orders     = df['Order_ID'].nunique()
total_units      = df['Quantity'].sum()
avg_order_value  = total_revenue / total_orders
avg_profit_margin = df['Profit_Margin'].mean()

print('=' * 45)
print('       📊 KEY PERFORMANCE INDICATORS')
print('=' * 45)
print(f'  Total Revenue      : ₹{total_revenue:,.0f}')
print(f'  Total Profit       : ₹{total_profit:,.0f}')
print(f'  Total Orders       : {total_orders}')
print(f'  Total Units Sold   : {total_units:,}')
print(f'  Avg Order Value    : ₹{avg_order_value:,.0f}')
print(f'  Avg Profit Margin  : {avg_profit_margin:.1f}%')
print('=' * 45)

## Step 5: Monthly Revenue Trend

In [ ]:
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = df.groupby('Month_Name')[['Sales_Amount','Profit']].sum().reindex(month_order)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(monthly.index, monthly['Sales_Amount'], marker='o', linewidth=2.5, 
        color='#2196F3', label='Revenue', markersize=8)
ax.plot(monthly.index, monthly['Profit'], marker='s', linewidth=2.5, 
        color='#4CAF50', label='Profit', markersize=8)
ax.fill_between(monthly.index, monthly['Sales_Amount'], alpha=0.1, color='#2196F3')
ax.set_title('Monthly Revenue & Profit Trend (2024)', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Month')
ax.set_ylabel('Amount (INR)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1e5:.1f}L'))
ax.legend()
plt.tight_layout()
plt.savefig('../images/monthly_revenue_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('📈 Peak month:', monthly['Sales_Amount'].idxmax())

## Step 6: Revenue by Category

In [ ]:
cat_rev = df.groupby('Category')['Sales_Amount'].sum().sort_values(ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart
colors = ['#2196F3', '#FF9800', '#4CAF50']
bars = ax1.bar(cat_rev.index, cat_rev.values, color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, cat_rev.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50000,
             f'₹{val/1e5:.1f}L', ha='center', fontweight='bold')
ax1.set_title('Revenue by Category', fontsize=14, fontweight='bold')
ax1.set_ylabel('Sales Amount (INR)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1e5:.0f}L'))

# Pie chart
ax2.pie(cat_rev.values, labels=cat_rev.index, autopct='%1.1f%%', colors=colors,
        startangle=90, textprops={'fontsize': 12})
ax2.set_title('Category Share (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../images/category_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print('🏆 Top Category:', cat_rev.idxmax())

## Step 7: Revenue by Region

In [ ]:
region_data = df.groupby('Region')[['Sales_Amount', 'Profit']].sum().sort_values('Sales_Amount', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
y = range(len(region_data))
bars1 = ax.barh(y, region_data['Sales_Amount'], height=0.4, label='Revenue', 
                color='#2196F3', align='center')
bars2 = ax.barh([i+0.4 for i in y], region_data['Profit'], height=0.4, 
                label='Profit', color='#4CAF50', align='center')
ax.set_yticks([i+0.2 for i in y])
ax.set_yticklabels(region_data.index)
ax.set_title('Revenue & Profit by Region', fontsize=14, fontweight='bold')
ax.set_xlabel('Amount (INR)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1e5:.0f}L'))
ax.legend()
plt.tight_layout()
plt.savefig('../images/region_sales.png', dpi=150, bbox_inches='tight')
plt.show()
print('🏆 Top Region:', region_data['Sales_Amount'].idxmax())

## Step 8: Top 10 Products

In [ ]:
top_products = df.groupby('Product')['Sales_Amount'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 6))
colors_gradient = plt.cm.Blues(np.linspace(0.4, 0.9, len(top_products)))[::-1]
bars = ax.bar(top_products.index, top_products.values, color=colors_gradient)
for bar, val in zip(bars, top_products.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20000,
            f'₹{val/1e5:.1f}L', ha='center', fontsize=9, fontweight='bold')
ax.set_title('Top 10 Products by Revenue', fontsize=14, fontweight='bold')
ax.set_xlabel('Product')
ax.set_ylabel('Total Sales (INR)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1e5:.0f}L'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../images/top_products.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9: Quarterly & Segment Analysis

In [ ]:
quarterly = df.groupby('Quarter')['Sales_Amount'].sum()
segment   = df.groupby('Customer_Segment')['Sales_Amount'].sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.bar(quarterly.index, quarterly.values, color=['#FF6B6B','#4ECDC4','#45B7D1','#96CEB4'], edgecolor='white')
ax1.set_title('Quarterly Revenue', fontsize=14, fontweight='bold')
ax1.set_ylabel('Sales Amount (INR)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1e5:.0f}L'))
for i, (q, v) in enumerate(quarterly.items()):
    ax1.text(i, v + 50000, f'₹{v/1e5:.1f}L', ha='center', fontweight='bold')

ax2.pie(segment.values, labels=segment.index, autopct='%1.1f%%',
        colors=['#FF6B6B','#4ECDC4','#45B7D1'], startangle=90)
ax2.set_title('Revenue by Customer Segment', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../images/quarterly_segment.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 10: Key Insights Summary

In [ ]:
print('=' * 55)
print('         🔍 KEY BUSINESS INSIGHTS SUMMARY')
print('=' * 55)
print(f'1. 📦 Top Category   : {df.groupby("Category")["Sales_Amount"].sum().idxmax()}')
print(f'2. 🌍 Top Region     : {df.groupby("Region")["Sales_Amount"].sum().idxmax()}')
print(f'3. 🏆 Best Product   : {df.groupby("Product")["Sales_Amount"].sum().idxmax()}')
print(f'4. 📅 Best Month     : {df.groupby("Month_Name")["Sales_Amount"].sum().idxmax()}')
print(f'5. 📊 Best Quarter   : {df.groupby("Quarter")["Sales_Amount"].sum().idxmax()}')
print(f'6. 👥 Top Segment    : {df.groupby("Customer_Segment")["Sales_Amount"].sum().idxmax()}')
print(f'7. 💰 Total Revenue  : ₹{df["Sales_Amount"].sum():,.0f}')
print(f'8. 📈 Profit Margin  : {(df["Profit"].sum()/df["Sales_Amount"].sum()*100):.1f}%')
print('=' * 55)